# Atividade 1 - Parte I: Comparação de Modelos e Otimização de Hiperparâmetros

**Dataset:** Car Evaluation (UCI) - 1728 instâncias, 6 atributos, 4 classes.

**Objetivo (Parte I):**
1. Comparar **Árvore de Decisão, Bagging, AdaBoost e Random Forest** usando validação cruzada estratificada (5 folds, com *shuffle*) e a métrica **F1-Macro** como critério decisor.
2. Selecionar o modelo com maior F1-Macro médio e realizar o **ajuste de hiperparâmetros** com `GridSearchCV` (5 folds estratificados, `scoring='f1_macro'`).
3. Comparar o desempenho do modelo **antes e depois** da otimização em um conjunto de teste independente (**holdout 70/30 estratificado**).

> **Cuidado com vazamento de dados:** a comparação dos modelos e o `GridSearchCV` usam **apenas o conjunto de treino (70%)**. O conjunto de teste (30%) só é usado na avaliação final.

In [ ]:
# Bibliotecas
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from etl.kaggle_extraction import run_extraction

SEED = 42
sns.set_style('whitegrid')

## 1. Carregamento dos dados

In [ ]:
# Extraindo Dataset do Kaggle
run_extraction()
print("Kaggle extraction completed.")

In [ ]:
# Criando o Dataframe (arquivo sem cabeçalho, por isso header=None)
colunas = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv("data/car_evaluation.csv", header=None, names=colunas)
df.head()

## 2. Análise exploratória

In [ ]:
# Avaliando nulos
print(df.isnull().sum())

In [ ]:
# Avaliando duplicados
print(df.duplicated().sum())

In [ ]:
df['class'].value_counts()

In [ ]:
# Gráfico de barras para evidenciar a distribuição das classes
ordem_classes = ['unacc', 'acc', 'good', 'vgood']
paleta = sns.color_palette('Blues', n_colors=len(ordem_classes))

plt.figure(figsize=(6, 4))
ax = sns.countplot(
    data=df, x='class', order=ordem_classes,
    hue='class', hue_order=ordem_classes, palette=paleta, legend=False
)

for p in ax.patches:
    ax.annotate(
        int(p.get_height()),
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

plt.title(f'Distribuição das Classes (n = {len(df)})')
plt.xlabel('Classe')
plt.ylabel('Contagem')
sns.despine()
plt.tight_layout()
plt.show()

## 3. Pré-processamento

Os atributos são categóricos **com ordem natural**, então foi aplicado *Ordinal Encoding* (mapeamento da Tabela 3 do relatório).

In [ ]:
# Ordinal Encoding
mapeamento = {
    'buying':   {'low': 1, 'med': 2, 'high': 3, 'vhigh': 4},
    'maint':    {'low': 1, 'med': 2, 'high': 3, 'vhigh': 4},
    'doors':    {'2': 2, '3': 3, '4': 4, '5more': 5},
    'persons':  {'2': 2, '4': 4, 'more': 5},
    'lug_boot': {'small': 1, 'med': 2, 'big': 3},
    'safety':   {'low': 1, 'med': 2, 'high': 3},
    'class':    {'unacc': 0, 'acc': 1, 'good': 2, 'vgood': 3}
}

# Aplicando o mapeamento no DataFrame
for coluna, mapa in mapeamento.items():
    df[coluna] = df[coluna].astype(str).map(mapa)

# Garantindo que nenhum valor ficou sem mapeamento
assert df.isnull().sum().sum() == 0, "Há valores não mapeados no encoding!"
df.head()

In [ ]:
# Separação em X e y
X = df.drop('class', axis=1)
y = df['class']

# Holdout estratificado 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print("\nProporção das classes (%):")
print(pd.DataFrame({
    'treino': y_train.value_counts(normalize=True).sort_index() * 100,
    'teste':  y_test.value_counts(normalize=True).sort_index() * 100,
}).round(3))

## 4. Comparação dos modelos (Stratified 5-Fold CV)

Os quatro algoritmos são avaliados com hiperparâmetros padrão (`random_state=42`), usando `StratifiedKFold(n_splits=5, shuffle=True)`. O critério decisor é o **F1-Macro médio**, que dá o mesmo peso a todas as classes (importante pelo desbalanceamento).

In [ ]:
modelos = {
    'Árvore de Decisão': DecisionTreeClassifier(random_state=SEED),
    'Bagging':           BaggingClassifier(random_state=SEED),
    'AdaBoost':          AdaBoostClassifier(random_state=SEED),
    'Random Forest':     RandomForestClassifier(random_state=SEED),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ['accuracy', 'f1_macro']

cv_resultados = {}
for nome, modelo in modelos.items():
    cv_resultados[nome] = cross_validate(
        modelo, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1
    )

resumo = pd.DataFrame({
    nome: {
        'Acurácia (média)':  res['test_accuracy'].mean(),
        'Acurácia (desvio)': res['test_accuracy'].std(),
        'F1-Macro (média)':  res['test_f1_macro'].mean(),
        'F1-Macro (desvio)': res['test_f1_macro'].std(),
    }
    for nome, res in cv_resultados.items()
}).T.sort_values('F1-Macro (média)', ascending=False)

resumo.round(4)

In [ ]:
# Distribuição do F1-Macro por fold
df_folds = pd.DataFrame({nome: res['test_f1_macro'] for nome, res in cv_resultados.items()})

plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df_folds, order=resumo.index, palette='Blues')
sns.stripplot(data=df_folds, order=resumo.index, color='black', size=4, alpha=0.6)
plt.title('F1-Macro por fold (Stratified 5-Fold CV)')
plt.ylabel('F1-Macro')
plt.xlabel('')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Seleção do melhor modelo (maior F1-Macro médio)
melhor_nome = resumo['F1-Macro (média)'].idxmax()
print(f"Melhor modelo na validação cruzada: {melhor_nome} "
      f"(F1-Macro médio = {resumo.loc[melhor_nome, 'F1-Macro (média)']:.4f})")

## 5. Ajuste de hiperparâmetros (GridSearchCV)

O `GridSearchCV` é aplicado ao modelo vencedor, com validação cruzada interna estratificada de 5 folds e `scoring='f1_macro'`. Há uma grade para cada algoritmo, então o código funciona qualquer que seja o vencedor da etapa anterior. A busca usa **somente o conjunto de treino**.

In [ ]:
param_grids = {
    'Árvore de Decisão': {
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_depth': [None, 5, 8, 10, 15],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': [None, 'balanced'],
    },
    'Bagging': {
        'n_estimators': [10, 50, 100, 200],
        'max_samples': [0.5, 0.8, 1.0],
        'max_features': [0.5, 0.8, 1.0],
        'bootstrap': [True, False],
    },
    'AdaBoost': {
        'n_estimators': [50, 100, 200, 400],
        'learning_rate': [0.1, 0.5, 1.0],
        'estimator': [DecisionTreeClassifier(max_depth=d, random_state=SEED) for d in (1, 2, 3, 5)],
    },
    'Random Forest': {
        'n_estimators': [100, 200, 400],
        'criterion': ['gini', 'entropy'],
        'max_depth': [None, 10, 15],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt', 'log2', None],
        'class_weight': [None, 'balanced'],
    },
}

modelo_base = modelos[melhor_nome]
grade = param_grids[melhor_nome]
n_combinacoes = int(np.prod([len(v) for v in grade.values()]))
print(f"Modelo: {melhor_nome} | {n_combinacoes} combinações x 5 folds = {n_combinacoes * 5} ajustes")

grid = GridSearchCV(
    estimator=modelo_base,
    param_grid=grade,
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs=-1,
    refit=True,
    verbose=1,
)
grid.fit(X_train, y_train)

print("\nMelhores hiperparâmetros:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nMelhor F1-Macro (CV interna): {grid.best_score_:.4f}")

In [ ]:
# Top 10 configurações da busca
cols = ['mean_test_score', 'std_test_score', 'rank_test_score', 'params']
pd.DataFrame(grid.cv_results_)[cols].sort_values('rank_test_score').head(10).reset_index(drop=True)

## 6. Avaliação final no holdout: antes vs. depois da otimização

- **Antes:** modelo vencedor com hiperparâmetros padrão, treinado em `X_train`.
- **Depois:** modelo retornado pelo `GridSearchCV` (`best_estimator_`), treinado em `X_train`.

Ambos são avaliados no mesmo conjunto de teste (30%), nunca visto durante a seleção ou o ajuste.

In [ ]:
class_names = ['unacc', 'acc', 'good', 'vgood']

# Modelo antes da otimização (hiperparâmetros padrão)
modelo_antes = modelos[melhor_nome].fit(X_train, y_train)
y_pred_antes = modelo_antes.predict(X_test)

# Modelo depois da otimização
modelo_depois = grid.best_estimator_
y_pred_depois = modelo_depois.predict(X_test)

comparacao = pd.DataFrame({
    'Antes (padrão)': [accuracy_score(y_test, y_pred_antes),
                       f1_score(y_test, y_pred_antes, average='macro')],
    'Depois (GridSearchCV)': [accuracy_score(y_test, y_pred_depois),
                              f1_score(y_test, y_pred_depois, average='macro')],
}, index=['Acurácia', 'F1-Macro'])
comparacao['Ganho'] = comparacao['Depois (GridSearchCV)'] - comparacao['Antes (padrão)']
comparacao.round(4)

In [ ]:
print("=== ANTES da otimização ===")
print(classification_report(y_test, y_pred_antes, target_names=class_names, digits=4))
print("=== DEPOIS da otimização ===")
print(classification_report(y_test, y_pred_depois, target_names=class_names, digits=4))

In [ ]:
# Matrizes de confusão
cm_antes = confusion_matrix(y_test, y_pred_antes)
cm_depois = confusion_matrix(y_test, y_pred_depois)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(cm_antes, display_labels=class_names).plot(
    ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title(f'{melhor_nome} - Antes da otimização')

ConfusionMatrixDisplay(cm_depois, display_labels=class_names).plot(
    ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title(f'{melhor_nome} - Depois da otimização')

plt.tight_layout()
plt.show()

In [ ]:
f1_antes = comparacao.loc['F1-Macro', 'Antes (padrão)']
f1_depois = comparacao.loc['F1-Macro', 'Depois (GridSearchCV)']

print(f"Modelo selecionado: {melhor_nome}")
if f1_depois > f1_antes:
    print(f"A otimização MELHOROU o desempenho no holdout (F1-Macro: {f1_antes:.4f} -> {f1_depois:.4f}).")
elif f1_depois < f1_antes:
    print(f"A otimização NÃO melhorou o desempenho no holdout (F1-Macro: {f1_antes:.4f} -> {f1_depois:.4f}).")
else:
    print(f"O desempenho no holdout permaneceu idêntico (F1-Macro: {f1_antes:.4f}).")